# Dataset loading and Dashboard

This notebook contains:
- Load functions for the dataset.
- Load function for the model.
- Generate interactive downprojection explorer.
- Interactive downprojection explorer for model score.
- Interactive downprojection exploerer for models tp/tn/fp/fn rates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torchxrayvision as xrv
import torch, torchvision
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Callable
import pandas as pd


# Projection algorithms
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

In [ ]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.widgets import RectangleSelector

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1 Constants and Configs

In [1]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed seed for reproducibility
RANDOM_STATE = 42

# From the ~65 000 images we only load a subset
MAX_NUMBER_OF_DATA = 10000

# or use 0.5
THRESHOLD = 0.7

MODEL_NAME = "densenet121-res224-chex"

# The path where the images for the dataset are stored. 
DATASET_IMAGE_PATH = './chest_data'
#DATASET_IMAGE_PATH = './nih_data_small'

#xrv.datasets.NIH_Dataset
#xrv.datasets.NIH_Google_Dataset
DATASET_XRAYTORCH_CLASS = xrv.datasets.CheX_Dataset
#DATASET_XRAYTORCH_CLASS = xrv.datasets.NIH_Google_Dataset

NameError: name 'torch' is not defined

In [ ]:
RNG = np.random.default_rng(RANDOM_STATE)

In [ ]:
HTML_OUTPUT_FOLDER = "html_plots/r0"

## 2 Load model

In [ ]:
def load_model(model_weights: str = "densenet121-res224-chex"):
    model = xrv.models.DenseNet(weights=model_weights)
    model.eval()
    return model.to(DEVICE)

In [ ]:
DENSENET121_RES244_CHEX_MODEL = load_model(model_weights=MODEL_NAME)

In [ ]:
MODEL_PATHOLOGIES = [str(path) for path in DENSENET121_RES244_CHEX_MODEL.pathologies if path != '']

## 3 Load and Prepare DATASET

In [ ]:
def load_dataset(image_path: str = './chest_data', datasetClass = xrv.datasets.CheX_Dataset, unique_patiens = True):
    transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
    d_nih = datasetClass(imgpath=image_path, transform=transform, views=["PA", "AP"], unique_patients=unique_patiens)
    print(f'found {len(d_nih)} images')
    return d_nih

We load the dataset and make sure the labels are align with the labels from the model. We also load only a subset of the images.

In [ ]:
XRAY_DATASET = load_dataset(DATASET_IMAGE_PATH, DATASET_XRAYTORCH_CLASS)
# We filterout pathologies which are not predicted by the model
DATASET_PATHOLOGIES = [str(path) for path in XRAY_DATASET.pathologies if path in MODEL_PATHOLOGIES]

# Align datasets to use the same pathologies
xrv.datasets.relabel_dataset(DATASET_PATHOLOGIES, XRAY_DATASET)
# Create a subset using indices
n = min(MAX_NUMBER_OF_DATA, len(XRAY_DATASET))
XRAY_DATASET = xrv.datasets.SubsetDataset(XRAY_DATASET, idxs=RNG.choice(len(XRAY_DATASET), size=n ,replace=True))
DATASET_LABEL_INDICES_MAP = {
        name: DATASET_PATHOLOGIES.index(name)
        for name in DATASET_PATHOLOGIES
}
DATASET_INDICES_LABEL_MAP = {DATASET_LABEL_INDICES_MAP[l]:l for l in DATASET_LABEL_INDICES_MAP}
print(f'Loaded {len(XRAY_DATASET)} images')
print(f'There are now {len(XRAY_DATASET.pathologies)} pathologies.')

In [ ]:
def get_patient_gender(dataset):
    metadata = getattr(dataset, "csv", None)

    def normalize_gender(value):
        value = str(value).strip().lower()

        if value == 'm' or value == 'male':
            return 0.0
        if value == 'f' or value == 'female':
            return 1.0
        return 2.0
    metadata_name = "Sex" if "Sex" in metadata else "Patient Gender"
    return metadata[metadata_name].map(normalize_gender).to_numpy()
    #return metadata["Patient Gender"].map(normalize_gender).to_numpy()

In [ ]:
XRAY_GENDER_LIST = get_patient_gender(XRAY_DATASET)

In [ ]:
def get_patient_age(dataset):
    metadata = getattr(dataset, "csv", None)

    def parse_age(value):
        value = str(value).strip().lower()

        try:
            return int(float(value))
        except ValueError:
            # TODO: handle proberly
            return -1
    metadata_name = "Age" if "Age" in metadata else "Patient Age"
    return metadata[metadata_name].map(parse_age).to_numpy()
    #return metadata["Patient Age"].map(parse_age).to_numpy()

In [ ]:
XRAY_AGE_LIST = get_patient_age(XRAY_DATASET)

In [ ]:
def get_dataset_views(dataset):
    metadata = getattr(dataset, "csv", None)

    def parse_age(value):
        value = str(value).strip().lower()

        try:
            return value
        except ValueError:
            # TODO: handle proberly
            return -1
    metadata_name = "AP/PA" if "AP/PA" in metadata else "View Position"
    return metadata[metadata_name].map(parse_age).to_numpy()

In [ ]:
XRAY_VIEW_LIST = get_dataset_views(XRAY_DATASET)

In [ ]:
def create_image_feature_space_and_label(dataset):

    source_indices = []
    target_indices = []

    for org_index, path in enumerate(XRAY_DATASET.pathologies):
        if path in DATASET_LABEL_INDICES_MAP:
            source_indices.append(org_index)
            target_indices.append(DATASET_LABEL_INDICES_MAP[path])

    source_indices = np.asarray(source_indices)
    target_indices = np.asarray(target_indices)


    image_features = []
    labels = []

    n_labels = len(DATASET_PATHOLOGIES)

    for i in range(len(dataset)):
        sample = dataset[i]

        # Image
        img = np.asarray(sample["img"], dtype=np.float32).squeeze()
        image_features.append(img)

        # Labels
        label = np.nan_to_num(
            np.asarray(sample["lab"]).reshape(-1)
        )

        adapted = np.zeros(n_labels, dtype=label.dtype)
        adapted[target_indices] = label[source_indices]

        labels.append(adapted)

    return np.asarray(image_features, dtype=np.float32), np.stack(labels)

In [ ]:
XRAY_IMAGE_FEATURES, XRAY_LABELS_LIST = create_image_feature_space_and_label(XRAY_DATASET)

In [ ]:
def get_model_to_dataset_indices(model = DENSENET121_RES244_CHEX_MODEL, dataset_label_indices_map=DATASET_LABEL_INDICES_MAP):
    return {
        model_idx: dataset_label_indices_map[pathology]
        for model_idx, pathology in enumerate(model.pathologies)
        if pathology in dataset_label_indices_map
    }   

In [ ]:
MODEL_TO_DATASET_INDICES_MAP = get_model_to_dataset_indices()
DATASET_TO_MODEL_INDICES_MAP = {MODEL_TO_DATASET_INDICES_MAP[key]:key for key in MODEL_TO_DATASET_INDICES_MAP}

In [ ]:
def print_dataset_stats(dataset=XRAY_DATASET, pathologies=DATASET_PATHOLOGIES, labels_list = XRAY_LABELS_LIST):
    print("Dataset statistics")
    print("=" * 50)

    print(f"Total number of samples: {len(dataset)}")
    print(f"Number of pathologies: {len(pathologies)}")
    print()

    for i, pathology in enumerate(pathologies):
        labels = labels_list[:, i]

        n_positive = np.sum(labels == 1)
        n_negative = np.sum(labels == 0)
        n_uncertain = np.sum(labels == -1)

        print(
            f"{pathology:25s} "
            f"positive: {n_positive:5d} | "
            f"negative: {n_negative:5d} | "
            f"uncertain: {n_uncertain:5d}"
        )

## 4 Prediction of the model

Let the model predict and store the values in the notebook.

In [ ]:
# make sure model predictions align with dataset 
def convert_to_dataset_label_list(model_prediction_score, dataset_labels_list = XRAY_LABELS_LIST,
                                  dataset_to_model_indices_map=DATASET_TO_MODEL_INDICES_MAP
                                  ):
    new_prection_list = np.zeros_like(dataset_labels_list)
    for selectedDatasetIndex in dataset_to_model_indices_map:
        selectedModelIndex = dataset_to_model_indices_map[selectedDatasetIndex]
        new_prection_list[:, selectedDatasetIndex] = model_prediction_score[:, selectedModelIndex]
    return new_prection_list

In [ ]:
def calculate_model_predictions_score(
    feature_space,
    model,
    batch_size=32
):
    model.eval()

    # Convertnto (N, 1, 224, 224)
    images = torch.as_tensor(feature_space, dtype=torch.float32)
    images = images.reshape(-1, 1, 224, 224)

    all_outputs = []

    with torch.no_grad():
        for start_idx in range(0, len(images), batch_size):
            end_idx = start_idx + batch_size

            batch = images[start_idx:end_idx].to(DEVICE)
            batch_outputs = model(batch)
            batch_outputs = batch_outputs.cpu().numpy()
            all_outputs.append(batch_outputs)

    # Combine all batches
    selected_outputs = np.concatenate(all_outputs, axis=0)
    return selected_outputs

In [ ]:
MODEL_SCORE_PREDICTIONS_CONVERTED = convert_to_dataset_label_list(calculate_model_predictions_score(XRAY_IMAGE_FEATURES,
                                                       DENSENET121_RES244_CHEX_MODEL))

In [ ]:
MODEL_PREDICTIONS = (MODEL_SCORE_PREDICTIONS_CONVERTED >= THRESHOLD).astype(int)

In [ ]:
def assert_input_valid(y_true: np.ndarray, y_scores: np.ndarray,):
    assert y_true.shape == y_scores.shape, (
        f"Shape mismatch: {y_true.shape} vs {y_scores.shape}"
    )

    assert y_true.ndim == 2, (
        f"Expected 2D arrays, got {y_true.ndim}D"
    )


## 5 Confusion matrix and ROC/PR curves

In [ ]:
def show_confusion_roc_pr_curves(
    y_true: np.ndarray,
    y_scores: np.ndarray,
    pathology_names: list[str],
    threshold=THRESHOLD):


    n_pathologies = len(pathology_names)

    assert_input_valid(y_true, y_scores)

    # create figure
    fig = make_subplots(
        rows=n_pathologies,
        cols=3,

        column_titles=[
            "Confusion Matrix",
            "ROC Curve",
            "Precision-Recall Curve"
        ],

        row_titles=pathology_names,

        horizontal_spacing=0.045,
        vertical_spacing=0.035,

        specs=[
            [
                {"type": "heatmap"},
                {"type": "xy"},
                {"type": "xy"}
            ]
            for _ in range(n_pathologies)
        ]
    )

    results = {}

    for i, pathology in enumerate(pathology_names):

        row = i + 1

        true = y_true[:, i]
        scores = y_scores[:, i]

        # calculate models 
        pred = (scores >= threshold).astype(int)

        cm = confusion_matrix(
            true,
            pred,
            labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        if len(np.unique(true)) != 2:
            raise RuntimeError('Cannot compute roc and pr curves for more than two type of labels!')

        # ROC
        fpr, tpr, roc_thresholds = roc_curve(
                true,
                scores
        )

        roc_auc = auc(
                fpr,
                tpr
        )

        # Precision-Recall
        precision, recall, pr_thresholds = (
            precision_recall_curve(
                true,
                scores
            )
        )

        ap = average_precision_score(
            true,
            scores
        )

        prevalence = np.mean(true)

        results[pathology] = {
            "confusion_matrix": cm,

            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,

            "fpr": fpr,
            "tpr": tpr,
            "roc_thresholds": roc_thresholds,
            "roc_auc": roc_auc,

            "precision": precision,
            "recall": recall,
            "pr_thresholds": pr_thresholds,
            "average_precision": ap,

            "prevalence": prevalence
        }

        # plot heatmap
        fig.add_trace(
            go.Heatmap(
                z=cm,

                x=[
                    "Predicted Negative",
                    "Predicted Positive"
                ],

                y=[
                    "Actual Negative",
                    "Actual Positive"
                ],

                colorscale="Blues",
                showscale=False,

                text=cm,
                texttemplate="%{text}",

                hovertemplate=(
                    f"<b>{pathology}</b><br>"
                    "Actual: %{y}<br>"
                    "Predicted: %{x}<br>"
                    "Count: %{z}<extra></extra>"
                )
            ),
            row=row,
            col=1
        )


        fig.add_trace(
                go.Scatter(
                    x=fpr,
                    y=tpr,

                    mode="lines",

                    name=f"{pathology} ROC",

                    line=dict(width=3),

                    hovertemplate=(
                        "FPR: %{x:.3f}<br>"
                        "TPR: %{y:.3f}<extra></extra>"
                    ),

                    showlegend=False
                ),
                row=row,
                col=2
        )

        # Random baseline
        fig.add_trace(
                go.Scatter(
                    x=[0, 1],
                    y=[0, 1],

                    mode="lines",

                    line=dict(
                        dash="dash",
                        color="gray"
                    ),

                    hoverinfo="skip",
                    showlegend=False
                ),
                row=row,
                col=2
        )

        fig.add_trace(
                go.Scatter(
                    x=recall,
                    y=precision,

                    mode="lines",

                    name=f"{pathology} PR",

                    line=dict(width=3),

                    hovertemplate=(
                        "Recall: %{x:.3f}<br>"
                        "Precision: %{y:.3f}<extra></extra>"
                    ),

                    showlegend=False
                ),
                row=row,
                col=3
            )

        # Prevalence baseline
        fig.add_hline(
                y=prevalence,

                line_dash="dash",
                line_color="gray",

                annotation_text=(
                    f"Prevalence = {prevalence:.3f}"
                ),

                annotation_position="top right",

                row=row,
                col=3
        )

    # axes
    for row in range(1, n_pathologies + 1):

        # confusion matrix
        fig.update_xaxes(
            title_text="Predicted",
            row=row,
            col=1
        )

        fig.update_yaxes(
            title_text="Actual",
            autorange="reversed",
            row=row,
            col=1
        )

        # roc
        fig.update_xaxes(
            title_text="False Positive Rate",
            range=[0, 1],
            row=row,
            col=2
        )

        fig.update_yaxes(
            title_text="True Positive Rate",
            range=[0, 1],
            row=row,
            col=2
        )

        # pr
        fig.update_xaxes(
            title_text="Recall",
            range=[0, 1],
            row=row,
            col=3
        )

        fig.update_yaxes(
            title_text="Precision",
            range=[0, 1],
            row=row,
            col=3
        )

    for i, pathology in enumerate(pathology_names):

        metrics = results[pathology]

        auc_value = metrics["roc_auc"]
        ap_value = metrics["average_precision"]

        if np.isnan(auc_value):
            auc_text = "AUC = N/A"
        else:
            auc_text = f"AUC = {auc_value:.3f}"

        if np.isnan(ap_value):
            ap_text = "AP = N/A"
        else:
            ap_text = f"AP = {ap_value:.3f}"


        fig.add_annotation(
            text=(
                f"<b>{pathology}</b>"
                f"<br><sup>{auc_text} | {ap_text}</sup>"
            ),

            x=-0.10,
            y=1 - (
                i + 0.5
            ) / n_pathologies,

            xref="paper",
            yref="paper",

            showarrow=False,

            xanchor="right",
            yanchor="middle",

            align="right"
        )

    fig.update_layout(

        title=dict(
            text=(
                "Multi-Label Chest X-Ray Model Evaluation"
                f"<br><sup>Classification threshold = {threshold}</sup>"
            ),
            x=0.5
        ),

        template="plotly_white",

        # More vertical space per pathology
        height=max(650, 420 * n_pathologies),

        # Slightly wider figure
        width=1500,

        margin=dict(
            l=150,
            r=40,
            t=100,
            b=50
        ),

        showlegend=False
    )

    fig.show()

    results["figure"] = fig

    return results

In [ ]:
results = show_confusion_roc_pr_curves(
    y_true=XRAY_LABELS_LIST,
    y_scores=MODEL_SCORE_PREDICTIONS_CONVERTED,
    pathology_names=DATASET_PATHOLOGIES,
    threshold=THRESHOLD
)

## 6 Downprojection

The Downprojector class helps to downproject a given image_feature_space and caches the result.

In [ ]:
class Downprojector:

    def __init__(self, image_features):
        self.projection_cache = {}
        self.image_features = image_features


    def get_feature_matrix(self, feature_space):
        return self.image_features

    def prepare_pca_features(self, feature_space):
        cache_key = f"{feature_space}_pca"
         
        if cache_key in self.projection_cache:
            return self.projection_cache[cache_key]
         
        X = self.get_feature_matrix(feature_space)
         
             # Keep at most 50 dimensions before expensive
             # non-linear dimensionality reduction.
        n_components = min(
                 50,
                 X.shape[0],
                 X.shape[1]
             )
         
        pca = PCA(
                 n_components=n_components,
                 random_state=RANDOM_STATE
        )
         
        X_pca = pca.fit_transform(X)
         
        self.projection_cache[cache_key] = {
                 "data": X_pca,
                 "pca": pca
             }
         
        return self.projection_cache[cache_key]

    def calculate_projection(self, feature_space: str, method: str,
                             target_labels=None,
                             label_name=None,
                             dataset_label_to_index_map = DATASET_LABEL_INDICES_MAP,):
        
        cache_key = (
                f"{feature_space}_{method}"
            ) if label_name is None else f"{feature_space}_{method}_{label_name}"
        
        if cache_key in self.projection_cache:
            return self.projection_cache[cache_key]


        if method == "PCA":

            X = self.get_feature_matrix(feature_space)

            reducer = PCA(
                n_components=2,
                random_state=RANDOM_STATE
            )

            embedding = reducer.fit_transform(X)
        elif method == "t-SNE":
        
            X_pca = self.prepare_pca_features(
                    feature_space
                )["data"]
        
                # t-SNE requires perplexity < number of samples.
            perplexity = min(
                    30,
                    max(5, (X_pca.shape[0] - 1) // 3)
                )
        
            reducer = TSNE(
                    n_components=2,
                    perplexity=perplexity,
                    init="pca",
                    learning_rate="auto",
                    random_state=RANDOM_STATE
                )
        
            embedding = reducer.fit_transform(
                    X_pca
                )
        elif method == "UMAP":
            X_pca = self.prepare_pca_features(
                    feature_space
                )["data"]

            n_neighbors = min(
                    15,
                    max(2, X_pca.shape[0] - 1)
                )

            reducer = umap.UMAP(
                    n_components=2,
                    n_neighbors=n_neighbors,
                    min_dist=0.1,
                    metric="euclidean",
                    random_state=RANDOM_STATE
                )

            embedding = reducer.fit_transform(
                    X_pca
                )
        # Supervised UMAP
        elif method == "S-UMAP":

            if target_labels is None:
                raise ValueError('Supervised UMAP needs target labels')

            if label_name is None:
                raise ValueError('Supervised UMAP needs a label name / pathologie')
            
            X_pca = self.prepare_pca_features(
                                feature_space
                            )["data"]
            
            n_neighbors = min(
                                15,
                                max(2, X_pca.shape[0] - 1)
                            )
            
            reducer = umap.UMAP(
                                n_components=2,
                                n_neighbors=n_neighbors,
                                min_dist=0.1,
                                metric="euclidean",
                                random_state=RANDOM_STATE
                            )

            target_index = dataset_label_to_index_map[label_name]

            Y = target_labels[:, target_index]
            
            embedding = reducer.fit_transform(
                                X_pca,
                                Y
                            )
        else:
            raise ValueError(
                f"Unknown projection method: {method}"
                )
        self.projection_cache[cache_key] = embedding
        return embedding

## 7 Generic viewer

In [ ]:
@dataclass
class XRayVisualizationData:

    feature_space: np.ndarray
    projection_method: str

    model_score_predictions: np.ndarray
    ground_truth: np.ndarray
    dataset_label_indices_map: dict

    dataset_gender_list: np.ndarray
    dataset_ages: np.ndarray

    selected_pathology: str
    threshold: float
    y_true: np.ndarray = None
    y_pred: np.ndarray = None
    y_scores: np.ndarray = None

    def __post_init__(self):
        label_idx = self.dataset_label_indices_map[self.selected_pathology]

        # Extract ground truth and prediction scores
        self.y_true = np.asarray(self.ground_truth[:, label_idx])
       
        
        self.y_scores = np.asarray(self.model_score_predictions[:, label_idx])

        self.y_pred = (self.y_scores >= self.threshold).astype(int)

In [ ]:
# Contract for a visualization function that the generic viewer can use
VisualizationFunction = Callable[
    [XRayVisualizationData],
    None
]

In [ ]:
# reusable plotly hover template for meta data
def create_hover_template(data: XRayVisualizationData, mask):
    
    customdata=np.column_stack((
                    data.y_true[mask],
                    data.y_scores[mask],
                    data.y_pred[mask],
                    np.asarray(data.dataset_ages)[mask],
                    np.asarray(data.dataset_gender_list)[mask]
                ))

    hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Component 1: %{x:.3f}<br>"
                    "Component 2: %{y:.3f}<br>"
                    "Ground truth: %{customdata[0]}<br>"
                    "Prediction score: %{customdata[1]:.3f}<br>"
                    "Binary prediction: %{customdata[2]}<br>"
                    "Age: %{customdata[3]} years<br>"
                    "Gender: %{customdata[4]}"
                    "<extra></extra>"
                )
    return customdata, hovertemplate

In [ ]:
def create_viewer(downprojector: Downprojector, visualization_fn: VisualizationFunction):

    plot_output = widgets.Output()

    progress_output = widgets.Output()

    plot_button = widgets.Button(
        description="Create selection plot",
        button_style="primary",
        icon="refresh"
    )

    pathology_dropdown = widgets.Dropdown(
        options=DATASET_PATHOLOGIES,
        value=DATASET_PATHOLOGIES[0],
        description="Pathology:",
        layout=widgets.Layout(width="350px")
    )

    projection_dropdown = widgets.Dropdown(
        options=[
            "PCA",
            "t-SNE",
            "UMAP",
            "S-UMAP"
        ],
        value="PCA",
        description="Projection:",
        layout=widgets.Layout(width="300px")
    )

    threshold_input = widgets.FloatSlider(
        value=0.7,
        min=0.0,
        max=1.0,
        step=0.1,
        description='Threshold',
        disabled=False
    )

    def create_selection_plot(_=None):

        with progress_output:
            clear_output(wait=True)
            pathology = pathology_dropdown.value
            projection_method = projection_dropdown.value

            target_labels = None
            label_name = None
            threshold = float(threshold_input.value)
            
            if projection_method == 'S-UMAP':
                target_labels = XRAY_LABELS_LIST
                label_name = pathology

            # Calculate the requested projection if it has not already
            # been calculated.
            embedding = downprojector.calculate_projection(
                "feature",
                projection_method,
                target_labels=target_labels,
                label_name = label_name
            )
        with plot_output:
            clear_output(wait=True)
     
            xray_data = XRayVisualizationData(
                feature_space=embedding,
                projection_method=projection_method,
                model_score_predictions=MODEL_SCORE_PREDICTIONS_CONVERTED,
                ground_truth=XRAY_LABELS_LIST,
                dataset_label_indices_map=DATASET_LABEL_INDICES_MAP,
                dataset_gender_list=XRAY_GENDER_LIST,
                dataset_ages=XRAY_AGE_LIST,
                selected_pathology=pathology,
                threshold = threshold
                )
            visualization_fn(xray_data)
            

    plot_button.on_click(create_selection_plot)

    display(
    widgets.VBox([
        widgets.HBox([
            projection_dropdown
        ]),
        widgets.HBox([
            pathology_dropdown
        ]),
        threshold_input,
        plot_button,
        progress_output,
        plot_output,
    ])
)

## 8 Interactive explorer for vizualization of downprojected spaces with ground truth and gender

Visualization function to show the embeddings of the downprojection with color and symbol cues to show gender and ground truth label.

In [ ]:
# The traces fn is used to only return the traces for the plot, but does create the actualy figure to plot in the notebook
def visualize_space_traces(
    data: XRayVisualizationData,
): 
    label_index = data.dataset_label_indices_map[data.selected_pathology]

    y_true = np.asarray(data.ground_truth[:, label_index])
    gender = np.asarray(data.dataset_gender_list)

    categories = {
        "Finding present · Male": (y_true == 1) & (gender == 0),
        "Finding present · Female": (y_true == 1) & (gender == 1),
        "Finding absent · Male": (y_true == 0) & (gender == 0),
        "Finding absent · Female": (y_true == 0) & (gender == 1)
    }

    colors = {
        "Finding present": "#D55E00",
        "Finding absent": "#0072B2"
    }


    symbols = {
        "Male": "circle",
        "Female": "triangle-up"
    }

    traces = []
    for category, mask in categories.items():

        status, sex = category.split(" · ")

        customdata, hovertemplate = create_hover_template(data, mask)
        
        traces.append(
            go.Scatter(
                x=data.feature_space[mask, 0],
                y=data.feature_space[mask, 1],
                mode="markers",
                name=category,
                marker=dict(
                    color=colors[status],
                    symbol=symbols[sex],
                    size=7,
                    opacity=0.75 if status == "Finding present" else 0.25
                ),
                customdata=customdata,
                hovertemplate=hovertemplate
            )
        )
    
    return traces

In [ ]:
def visualize_space(
    data: XRayVisualizationData,
): 
    projection_method = data.projection_method
    
    fig = go.Figure()
    traces = visualize_space_traces(data)
    fig.add_traces(traces)
    fig.update_layout(
        title=dict(
            text=f"Distribution of Chest X-ray Embeddings by {data.selected_pathology} and Gender",
            x=0.5
        ),
        xaxis_title=f"{projection_method} 1",
        yaxis_title=f"{projection_method} 2",
        template="plotly_white",
        width=1000,
        height=750,
        legend=dict(
            title="Pathology Status and Gender",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        hovermode="closest"
    )
    
    fig.show()

In [ ]:
DOWN_PROJECTOR = Downprojector(XRAY_IMAGE_FEATURES.reshape(-1, 224* 224))

In [ ]:
create_viewer(DOWN_PROJECTOR, visualize_space)

## 9 Interactive explorer for the prediction score of the model

In [ ]:
def visualize_space_prediction_scores_traces(data: XRayVisualizationData):

    label_index = data.dataset_label_indices_map[data.selected_pathology]
      
    y_true = np.asarray(data.ground_truth[:, label_index])
    gender = np.asarray(data.dataset_gender_list)
      
    categories = {
              "Finding present · Male": (y_true == 1) & (gender == 0),
              "Finding present · Female": (y_true == 1) & (gender == 1),
              "Finding absent · Male": (y_true == 0) & (gender == 0),
              "Finding absent · Female": (y_true == 0) & (gender == 1)
    }

    
    symbols = {
              "Male": "circle",
              "Female": "triangle-up"
      }
      
    traces = []
      
    for i, (category, mask) in enumerate(categories.items()):
      
        status, sex = category.split(" · ")
      
        customdata, hovertemplate = create_hover_template(data, mask)
  
        scores = data.y_scores[mask]
              
        traces.append(
                  go.Scatter(
                      x=data.feature_space[mask, 0],
                      y=data.feature_space[mask, 1],
                      mode="markers",
                      name=category,
                      marker=dict(
                          color=scores,
                          colorscale="Viridis",
                          cmin=0,
                          cmax=1,
                          showscale=(i == 0),  # Show only one color bar
                          colorbar=dict(
                              title="Model score",
                              thickness=18,
                              len=0.75
                          ) if i == 0 else None,
                          symbol=symbols[sex],
                          size=7,
                          opacity=0.75 if status == "Finding present" else 0.25
                      ),
                      customdata=customdata,
                      hovertemplate=hovertemplate
                  )
              )
      

    return traces

In [ ]:
def visualize_space_prediction_scores(data: XRayVisualizationData):
    projection_method = data.projection_method

    fig = go.Figure()
    traces = visualize_space_prediction_scores_traces(data)
    fig.add_traces(traces)
    
    fig.update_layout(
        title=dict(
            text=f"{projection_method} Projection Colored by {data.selected_pathology} Model Score",
            x=0.5
        ),
        xaxis_title=f"{projection_method} 1",
        yaxis_title=f"{projection_method} 2",
        template="plotly_white",
        width=1000,
        height=750,
        legend=dict(
            title="Pathology Status and Gender",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        hovermode="closest"
    )
    
    fig.show()

In [ ]:
create_viewer(DOWN_PROJECTOR, visualize_space_prediction_scores)

## 10 Interactive explorer to visualize the TP/TN/FP/FN 

In [ ]:
def visualize_classification_outcome_traces(
    data: XRayVisualizationData
):
        label_idx = data.dataset_label_indices_map[data.selected_pathology]
    
        y_true = data.y_true
        y_scores = np.asarray(data.model_score_predictions[:, label_idx])
    
        y_pred = (y_scores >= data.threshold).astype(int)
    
        categories = {
            "True Positive": (y_true == 1) & (y_pred == 1),
            "True Negative": (y_true == 0) & (y_pred == 0),
            "False Positive": (y_true == 0) & (y_pred == 1),
            "False Negative": (y_true == 1) & (y_pred == 0)
        }
    
        colors = {
            "True Positive": "#009E73",
            "True Negative": "#0072B2",
            "False Positive": "#D55E00",
            "False Negative": "#CC79A7"
        }
    
        traces = []
    
        for category, mask in categories.items():
    
            customdata , hovertemplate = create_hover_template(data, mask)
    
            traces.append(
                go.Scatter(
                    x=data.feature_space[mask, 0],
                    y=data.feature_space[mask, 1],
                    mode="markers",
                    name=category,
                    marker=dict(
                        color=colors[category],
                        size=7 if "False" not in category else 9,
                        opacity=0.8,
                        line=dict(
                            width=0.5,
                            color=colors[category]
                        )
                    ),
                    customdata=customdata,
    
                    hovertemplate=hovertemplate
                )
            )
    
        return traces

In [ ]:
def visualize_classification_outcome(
    data: XRayVisualizationData
):

    projection_method = data.projection_method

   
    fig = go.Figure()
    traces = visualize_classification_outcome_traces(data)
    fig.add_traces(traces)

    fig.update_layout(
        title=dict(
            text=f"Classification Outcomes in {projection_method} Embedding Space – {data.selected_pathology}",
            x=0.5
        ),
        xaxis_title=f"{projection_method} 1",
        yaxis_title=f"{projection_method} 2",
        template="plotly_white",
        width=1000,
        height=750,
        legend=dict(
            title="Classification Outcome",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        ),
        hovermode="closest",
    )

    fig.show()

In [ ]:
create_viewer(DOWN_PROJECTOR, visualize_classification_outcome)

## 11 Make subplot to easier compare visualization for both human and AI

In [ ]:
def create_subplot_for_visualization_fn(
    downprojector: Downprojector,
    projection_method: str,
    visualization_fns: list[VisualizationFunction],
    visualization_names: list[str],
    model_score_predictions,
    ground_truth,
    dataset_label_indices_map,
    dataset_gender_list=None,
    dataset_ages=None,
    pathologies=DATASET_PATHOLOGIES,
    threshold=THRESHOLD,
    plotHTML=True,
    output_file="pathology_comparison.html"
):

    n_rows = len(pathologies)
    n_cols = len(visualization_fns)

    subplot_titles = [
        f"{pathology} - {name}"
        for pathology in pathologies
        for name in visualization_names
    ]

    print(subplot_titles)

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.06,
        vertical_spacing=0.04,
        row_heights=[1 / n_rows] * n_rows,
        column_widths=[1 / n_cols] * n_cols
    )

    for row, pathology in enumerate(pathologies, start=1):

        # Embeddings are cached
        embedding = downprojector.calculate_projection(
                "feature",
                projection_method,
                target_labels=ground_truth,
                label_name=pathology
        )

        xray_data = XRayVisualizationData(
            feature_space=embedding,
            projection_method=projection_method,
            model_score_predictions=model_score_predictions,
            ground_truth=ground_truth,
            dataset_label_indices_map=dataset_label_indices_map,
            dataset_gender_list=dataset_gender_list,
            dataset_ages=dataset_ages,
            selected_pathology=pathology,
            threshold=threshold
        )

        for col, visualization_fn in enumerate(visualization_fns, start=1):

            traces = visualization_fn(xray_data)
    
            for trace in traces:
                trace.legendgroup = trace.name
                trace.showlegend = (row == 1)

                fig.add_trace(
                    trace,
                    row=row,
                    col=col
                )

    fig.update_layout(
        title=dict(
            text=f"{projection_method} - Comparison Across Pathologies",
            x=0.5,
            y=0.99
        ),
        template="plotly_white",

        # Figure dimensions
        height=max(400 * n_rows, 800),
        width=max(700 * n_cols, 1400),

        margin=dict(
            l=50,
            r=120,
            t=90,
            b=50
        ),

        hovermode="closest",

        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.01
        )
    )

    fig.update_xaxes(
        showgrid=True,
        zeroline=False,
        automargin=True
    )

    fig.update_yaxes(
        showgrid=True,
        zeroline=False,
        automargin=True
    )
    if plotHTML:
        fig.write_html(
            output_file,
            include_plotlyjs="cdn",
            full_html=True
        )
    else:
        fig.show()

In [ ]:
visualization_functions = [
    visualize_space_prediction_scores_traces,
    visualize_classification_outcome_traces,
]

visualization_names = [
    "Model Confidence",
    "Classification outcomes"
]

create_subplot_for_visualization_fn(
        downprojector=DOWN_PROJECTOR,
        projection_method="S-UMAP",
        visualization_fns=visualization_functions,
        visualization_names= visualization_names,
        model_score_predictions=MODEL_SCORE_PREDICTIONS_CONVERTED,
        ground_truth=XRAY_LABELS_LIST,
        dataset_label_indices_map=DATASET_LABEL_INDICES_MAP,
        dataset_gender_list=XRAY_GENDER_LIST,
        dataset_ages=XRAY_AGE_LIST,
        threshold=THRESHOLD,
        plotHTML=True
    )